In [1]:
import sounddevice as sd
import numpy as np
import librosa

In [2]:
fs = 16000
duration = 2

commands = ["start", "stop"]


# Record audio from microphone
def record_audio():
    print("Recording...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    print("Recording Finished")
    return audio.flatten()


# Extract MFCC features
def extract_mfcc(signal):
    mfcc = librosa.feature.mfcc(y=signal, sr=fs, n_mfcc=13)
    return mfcc.T

In [3]:
# DTW distance calculation
def dtw_distance(seq1, seq2):
    n, m = len(seq1), len(seq2)

    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = np.linalg.norm(seq1[i - 1] - seq2[j - 1])

            dtw[i, j] = cost + min(
                dtw[i - 1, j],
                dtw[i, j - 1],
                dtw[i - 1, j - 1]
            )

    return dtw[n, m]


In [4]:
# Record template commands
templates = {}

for cmd in commands:
    print(f"Say '{cmd}' now")
    audio = record_audio()
    templates[cmd] = extract_mfcc(audio)


# Record test command
print("Say a command for recognition")
test_audio = record_audio()
test_mfcc = extract_mfcc(test_audio)


# Compare with templates
distances = {}

for cmd in commands:
    distances[cmd] = dtw_distance(test_mfcc, templates[cmd])


# Predict command
predicted_command = min(distances, key=distances.get)

print("Predicted Command:", predicted_command)

Say 'start' now
Recording...
Recording Finished


c:\Annaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Annaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Annaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Say 'stop' now
Recording...
Recording Finished
Say a command for recognition
Recording...
Recording Finished
Predicted Command: stop
